# Nigerian LGA OSM Extractor — Worked Example: Akure North & Akure South

## 1. Introduction

This notebook is the reference example for the **Nigerian LGA OSM Extractor**
(`lga_extractor`), a reusable Python tool submitted to **Map<>kathon 2026**
(Lightweight Tool or Demo track). It demonstrates the full pipeline end to
end: resolving an LGA boundary, extracting six OSM feature layers, cleaning
and exporting them, and generating a polished visual preview.

The two LGAs used here — **Akure North** and **Akure South** (Ondo State,
Nigeria) — are also the study areas for the companion submission,
*"Mapping the Gap: Health & Education Accessibility in Akure"* (OSM
Dashboard or Analysis track), so the outputs produced in this notebook feed
directly into that project's `01_data_extraction.ipynb`.

### Key concept: what does "extracting OSM data for an LGA" actually involve?

Three distinct GIS operations, done for you inside `extract_lga()`:

1. **Boundary resolution (geocoding a place name to a shape, not a point)**
   — "Akure North, Ondo, Nigeria" needs to become an actual polygon before
   anything else can happen. This works by querying OSM's own
   administrative-boundary relations (places tagged as an `admin_level`
   boundary) — different from typical "geocoding," which usually resolves
   a name to a single point, not a full shape.
2. **Tag-based feature querying** — OSM doesn't have a "roads table" the
   way a traditional GIS database might; instead, every real-world feature
   is described by free-form key=value **tags** (e.g. `highway=primary` for
   a major road, `amenity=hospital` for a hospital). Extracting "all roads"
   means querying for everything carrying any `highway=*` tag within the
   boundary — this project's `DEFAULT_TAG_CONFIG` (see Section 2) defines
   exactly which tags map to which of our six output layers.
3. **Reprojection to a consistent, metric CRS** — OSM data is stored in
   WGS84 (plain latitude/longitude degrees), which isn't suitable for
   measuring real distances or areas. Every layer gets reprojected to
   EPSG:32631 (UTM Zone 31N) during cleaning, so that later distance-based
   analysis (like the companion project's accessibility scoring) works
   correctly.

### What this notebook covers

1. Introduction
2. Imports
3. Configuration — which LGAs, which tags, where outputs go
4. Data loading (extraction) — Akure North, then Akure South
5. Processing — validation and side-by-side comparison
6. Visualization — quick matplotlib check, then a polished kepler.gl preview
7. Export — inspecting the run log for reproducibility
8. Summary and next steps

### Prerequisites

- The `lga_extractor` package (this repository), either installed in
  editable mode (`pip install -e .`) or importable via `sys.path`.
- Internet access (the extraction step queries the live OpenStreetMap
  Overpass API through OSMnx).
- Optional: `keplergl` for the polished preview map in Section 6.

## 2. Imports

The one import that matters here is `extract_lga` — the single function
that wraps the entire pipeline described in the Introduction. `DEFAULT_TAG_CONFIG`
is also imported so we can inspect (and optionally extend) exactly which
OSM tags map to which output layer.

### 2.1 Environment setup

Run the cell below first. It detects whether you're in Google Colab or a
local environment and adapts automatically:

- **In Colab:** mounts Google Drive, installs dependencies, and works
  directly out of your Drive folder (recommended: create a Drive folder
  named **`LGA OSM Extractor`** at the root of My Drive and place this
  repository's contents inside it — that's the path assumed below).
- **Locally:** assumes you've already run `pip install -r requirements.txt`
  (or `conda env create -f environment.yml`) and are running this notebook
  from inside the repository's `examples/` folder.

Because Colab sessions are ephemeral, working directly inside a mounted
Drive folder (rather than cloning into `/content/` each time) means your
extracted outputs, run logs, and any edits persist automatically between
sessions without needing to push to GitHub every time.

In [ ]:
# --- Environment setup ---
import sys, os

IN_COLAB = "google.colab" in sys.modules

# Update this to match the Drive folder name you created, if different.
DRIVE_FOLDER_NAME = "LGA OSM Extractor"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    REPO_DIR = f"/content/drive/MyDrive/{DRIVE_FOLDER_NAME}"
    if not os.path.exists(REPO_DIR):
        raise FileNotFoundError(
            f"Expected '{REPO_DIR}' to exist in your Google Drive.\n"
            f"Create a folder named '{DRIVE_FOLDER_NAME}' at the root of "
            f"My Drive and upload this repository's contents into it, "
            f"then re-run this cell."
        )

    %cd {REPO_DIR}
    !pip install osmnx geopandas shapely fiona pandas leafmap keplergl --quiet
    print(f"Running in Colab. Working directory: {os.getcwd()}")
else:
    print("Running locally. Assuming dependencies are already installed "
          "and this notebook is being run from the repository's examples/ folder.")


### 2.2 Package imports

(Combined with Section 3's configuration below, since the import and the
values it exposes — `DEFAULT_TAG_CONFIG` — are used together.)

## 3. Configuration

All the parameters that vary between runs are collected here in one place,
rather than scattered through the notebook. Change these if you want to
extract a different LGA, or a different set of layers.

In [ ]:
import sys
if not IN_COLAB:
    sys.path.append("..")  # so `import lga_extractor` works when run from examples/

from lga_extractor import extract_lga, DEFAULT_TAG_CONFIG

# --- Study areas for this notebook ---
STUDY_AREAS = [
    {"lga_name": "Akure North", "state_name": "Ondo", "output_dir": "../output/akure_north"},
    {"lga_name": "Akure South", "state_name": "Ondo", "output_dir": "../output/akure_south"},
]

# --- Tag configuration ---
# DEFAULT_TAG_CONFIG covers roads, buildings, waterways, land use, health
# facilities, and schools (see lga_extractor/layers.py). Override or extend
# it here if you need additional feature types, e.g.:
#
# from lga_extractor.layers import DEFAULT_TAG_CONFIG
# tag_config = {**DEFAULT_TAG_CONFIG, "markets": {"amenity": "marketplace"}}
tag_config = DEFAULT_TAG_CONFIG

print("Configured study areas:")
for area in STUDY_AREAS:
    print(f"  - {area['lga_name']}, {area['state_name']} -> {area['output_dir']}")
print()
print("Configured layers:", list(tag_config.keys()))


## 4. Data Loading

### 4.1 Extraction — Akure North

`extract_lga()` runs the full pipeline in one call:

1. **Boundary resolution** — geocodes "Akure North, Ondo, Nigeria" against
   OSM's administrative boundary relations.
2. **Layer extraction** — queries each configured tag set (roads, buildings,
   waterways, land use, health facilities, schools) within that boundary.
3. **Cleaning** — reprojects to EPSG:32631, repairs invalid geometries,
   removes duplicates, standardizes the attribute schema.
4. **Export** — writes GeoJSON and Shapefile per layer.
5. **Logging** — writes `run_log.json` recording exactly what was queried
   and when, for reproducibility.

This step requires internet access and can take anywhere from under a
minute to several minutes depending on Overpass server load and how densely
mapped the area is.

In [ ]:
north_config = STUDY_AREAS[0]

result_north = extract_lga(
    lga_name=north_config["lga_name"],
    state_name=north_config["state_name"],
    output_dir=north_config["output_dir"],
    tag_config=tag_config,
)

print(f"Boundary resolved via: {result_north['boundary_source']}")
print(f"Output directory: {result_north['output_dir']}")
print(f"Run log: {result_north['run_log']}")
if result_north["warnings"]:
    print("\nWarnings:")
    for w in result_north["warnings"]:
        print(f"  - {w}")


### 4.2 Extraction — Akure South

Repeating the same process for Akure South. Because `extract_lga()` is
fully parameterized, this is identical to the previous step aside from the
LGA name and output directory — this repeatability across any Nigerian LGA
is the core value proposition of the tool.

In [ ]:
south_config = STUDY_AREAS[1]

result_south = extract_lga(
    lga_name=south_config["lga_name"],
    state_name=south_config["state_name"],
    output_dir=south_config["output_dir"],
    tag_config=tag_config,
)

print(f"Boundary resolved via: {result_south['boundary_source']}")
print(f"Output directory: {result_south['output_dir']}")
print(f"Run log: {result_south['run_log']}")
if result_south["warnings"]:
    print("\nWarnings:")
    for w in result_south["warnings"]:
        print(f"  - {w}")


## 5. Processing

### 5.1 Validation — Akure North

Before moving on, it's worth confirming the extraction actually produced
usable data rather than silently returning empty layers (which can happen
if a boundary resolves to the wrong area, or if OSM coverage for a tag is
genuinely sparse in that location).

In [ ]:
import geopandas as gpd

print(f"{'Layer':<20}{'Feature count':>15}{'CRS':>15}")
print("-" * 50)
for layer_name, paths in result_north["exported"].items():
    if layer_name.startswith("_"):  # skip metadata keys like _skipped, _split_layers
        continue
    gdf = gpd.read_file(paths["geojson"])
    print(f"{layer_name:<20}{len(gdf):>15}{str(gdf.crs):>15}")

if result_north["exported"].get("_skipped"):
    print(f"\nSkipped (no features found): {result_north['exported']['_skipped']}")


### 5.2 Validation — Akure South

In [ ]:
print(f"{'Layer':<20}{'Feature count':>15}{'CRS':>15}")
print("-" * 50)
for layer_name, paths in result_south["exported"].items():
    if layer_name.startswith("_"):  # skip metadata keys like _skipped, _split_layers
        continue
    gdf = gpd.read_file(paths["geojson"])
    print(f"{layer_name:<20}{len(gdf):>15}{str(gdf.crs):>15}")

if result_south["exported"].get("_skipped"):
    print(f"\nSkipped (no features found): {result_south['exported']['_skipped']}")


### 5.3 Side-by-side comparison

A quick sanity check comparing feature counts between the two LGAs. Large,
unexpected asymmetries (e.g. one LGA having zero health facilities while
the neighboring one has a dozen) are often a signal worth investigating —
either a genuine OSM data gap (relevant to the completeness analysis in the
companion project) or a boundary resolution issue worth double-checking.

In [ ]:
import pandas as pd

comparison_rows = []
for name, result in [("Akure North", result_north), ("Akure South", result_south)]:
    row = {"LGA": name}
    for layer_name, paths in result["exported"].items():
        if layer_name.startswith("_"):  # skip metadata keys like _skipped, _split_layers
            continue
        gdf = gpd.read_file(paths["geojson"])
        row[layer_name] = len(gdf)
    comparison_rows.append(row)

comparison_df = pd.DataFrame(comparison_rows).set_index("LGA")
comparison_df


## 6. Visualization

### 6.1 Quick visual check (matplotlib)

A fast, dependency-light sanity check: plot one layer to confirm the
geometry looks like a real road network rather than, say, a single point or
an empty/malformed shape.

In [ ]:
roads_north = gpd.read_file(f"{north_config['output_dir']}/roads.geojson")
ax = roads_north.plot(figsize=(8, 8), linewidth=0.5, color="black")
ax.set_title(f"Akure North — Road Network ({len(roads_north)} segments)")
ax.set_axis_off()


### 6.2 Polished preview map (kepler.gl)

For a nicer, all-layers-at-once view — useful for sanity-checking an
extraction visually, or for a quick shareable preview without needing GIS
software — the tool includes a `build_preview_map()` helper. It loads
whichever layers exist for the LGA, applies a default style
(`kepler_config_lga_preview.json`), and can save a standalone HTML file
(data + viewer bundled in one file) that opens in any browser.

This is a visual convenience layer only, not an analysis tool. Requires
`pip install keplergl` (already installed by the Colab setup cell above; if
running locally and it's missing, install it before running this cell).

In [ ]:
from lga_extractor import build_preview_map

preview_map_north = build_preview_map(
    output_dir=north_config["output_dir"],
    html_out="../visuals/akure_north_preview.html",
)
preview_map_north


In [ ]:
preview_map_south = build_preview_map(
    output_dir=south_config["output_dir"],
    html_out="../visuals/akure_south_preview.html",
)
preview_map_south


## 7. Export

### 7.1 Inspecting the run log (reproducibility)

Every extraction writes a `run_log.json` recording exactly what was
queried, when, and with what tag configuration. This directly supports the
reproducibility criterion in the Map<>kathon judging rubric — anyone
reviewing this submission can see precisely how each dataset was produced
and reproduce it themselves.

In [ ]:
import json

for name, result in [("Akure North", result_north), ("Akure South", result_south)]:
    print(f"--- {name} run log ---")
    with open(result["run_log"]) as f:
        log = json.load(f)
    print(json.dumps(log, indent=2))
    print()


**Outputs produced:**

```
output/akure_north/   {roads, buildings, waterways, landuse,
                        health_facilities, schools}.geojson + .shp, run_log.json
output/akure_south/    (same structure)
visuals/akure_north_preview.html
visuals/akure_south_preview.html
```

## 8. Summary

This notebook has:

- Extracted six OSM feature layers (roads, buildings, waterways, land use,
  health facilities, schools) for both Akure North and Akure South
- Validated feature counts and CRS for each layer
- Produced a quick matplotlib sanity check and a polished kepler.gl preview
  for each LGA
- Confirmed reproducibility by inspecting the logged Overpass query
  parameters for each run

**Where this feeds next:** these exact outputs are consumed directly by
the companion repository's `notebooks/01_data_extraction.ipynb` in
*"Mapping the Gap: Health & Education Accessibility in Akure"*, which
builds the accessibility and completeness analysis on top of this data.

**To extract a different LGA:** change the `STUDY_AREAS` list in Section 3
and re-run — no other code changes are required, since the pipeline is
fully parameterized by LGA name and state.